In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://www.walkerplus.com/event_list/ar0313/sc309880d/',
   'title': '東京駅(東京都)周辺のイベント - ウォーカープラス',
   'content': '開催中 2025年12月20日(土)～2026年2月22日(日). 京橋駅(東京都), 宝町駅(東京都), 日本橋駅(東京都), 銀座一丁目駅(東京都), 東京駅(東京都). CREATIVE MUSEUM TOKYO(クリエイティブ ミュージアム トウキョウ). 日比谷駅(東京都), 有楽町駅(東京都), 東京駅(東京都), 京橋駅(東京都). 開催中 2026年1月2日(金)～3月22日(日). 二重橋前駅(東京都), 有楽町駅(東京都), 東京駅(東京都), 日比谷駅(東京都), 銀座一丁目駅(東京都). * 歴史リアル謎解きゲーム「謎の城」in 日本橋「発明家／人斬り-平賀源内-」. 開催中 2025年9月1日(月)～2026年3月1日(日). 日本橋駅(東京都), 京橋駅(東京都), 東京駅(東京都), 宝町駅(東京都), 三越前駅(東京都). * MIDTOWN YAESU CHRISTMAS 2025 (ミッドタウン八重洲クリスマス2025). 終了間近 2025年11月13日(木)～2026年2月15日(日). 京橋駅(東京都), 東京駅(東京都), 宝町駅(東京都), 日本橋駅(東京都), 銀座一丁目駅(東京都). * 丸の内から御食国「敦賀・若狭」へ 美し物-うましもの-グルメフェア. 開催中 2026年2月2日(月)～21日(土). 二重橋前駅(東京都), 有楽町駅(東京都), 東京駅(東京都), 日比谷駅(東京都). 終了間近 2025年11月13日(木)～2026年2月15日(日). 二重橋前〈丸の内〉駅(東京都), 東京駅(東京都), 有楽町駅(東京都), 日比谷駅(東京都), 銀座一丁目駅(東京都). * Masking Tape Jamboree in KITTE2026. 終了間近 2026年2月11日(水)～13日(金). 東京駅(東京都), 二重橋前駅(東京都), 有楽町駅(東京都), 京橋駅(東京都), 銀座一丁目駅(東京都). 東京駅(東京都), 二重橋前駅

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- **東京都**の面積は約2,194 km²です。
- **沖縄県**の面積は約2,271 km²です。

したがって、沖縄県の方が東京都よりも広い面積を持っています。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近の東京駅周辺でのイベント情報は以下の通りです。

1. [2023年2月最新] 東京駅イベント情報
   - 記事では、東京駅の最新のイベントや発信情報が掲載されています。特に「東京ステーションギャラリー110周年」などが注目です。
   - 詳細はこちら: [東京駅周辺イベント情報](https://bestcalendar.jp/events/%E6%9D%B1%E4%BA%AC%E9%A7%85)

2. [東京駅周辺のイベント情報](https://www.walkerplus.com/event_list/ar0313/sc309880d/)
   - 2025年から2026年にかけての大規模なイベントや、日常的に楽しめるイベントが紹介されています。

3. [Tokyo Illumilia 2025-2026](https://www.enjoytokyo.jp/event/list/sta200101/)
   - テーマは「イルミネーション」で、2025年から2026年にかけて開催予定のイベント内容が紹介されています。

これらの情報をもとに、東京駅周辺でのイベントを楽しんでください！


In [11]:
# チャットボットへの組み込み
tools = define_tools()

messages = []

while(True):
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip() == "":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # メッセージ履歴が8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！今日はどんなことをお手伝いできますか？


'質問:東北6県は？'

東北6県は以下の通りです：

1. 青森県
2. 岩手県
3. 宮城県
4. 秋田県
5. 山形県
6. 福島県

これらの県は日本の北東部に位置しています。


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のお土産についての情報をいくつかご紹介します。

1. **牛たん** - 宮城県の名物である牛たんは、塩味で焼いたり、スモークして食べるスタイルが人気です。

2. **ずんだ餅** - ずんだとは、枝豆をすり潰して作る餡で、宮城の名物として有名です。甘みのあるずんだ餡をかけたお餅が一般的です。

3. **萩の月** - 宮城県仙台の有名な生菓子で、しっとりとした皮にクリームが入っています。このお土産は大変人気があります。

4. **仙台かすてら** - 繊細な口当たりと豊かな甘みで、多くの人々に愛されているカステラです。

5. **喜久福** - 人気のある和菓子で、皮の中に様々な味の餡が入っています。特に、抹茶や黒ごま味が人気です。

6. **宮城県の酒** - 地元で生産されている日本酒もお土産におすすめです。地元産の米を使用した酒から、多様な風味のものまであります。

これらのお土産は、宮城県を訪れた際にお持ち帰りするのに最適です。特に、地元の特色が感じられる品々がたくさんありますので、ぜひお試しください！

---ご利用ありがとうございました！---


## 課題 メンターにチャットボットをレビューしてもらおう

### 概要
- 元のコードは`messages`を作成・更新しているにもかかわらず利用していないので、`messages`を投げるように修正

In [12]:
# 言語モデルへの質問を行う関数
def ask_llm(messages, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    return response

In [13]:
# ユーザーからの質問を処理する関数
def process_chat(messages, tools):
    response = ask_llm(messages, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        question = messages[-1]["content"]
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [14]:
# チャットボットへの組み込み
tools = define_tools()

messages = []

while(True):
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip() == "":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # メッセージ履歴が8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_chat(messages, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！どのようにお手伝いできますか？


'質問:東北6県は？'

東北地方の6県は以下の通りです：

1. 青森県
2. 岩手県
3. 宮城県
4. 秋田県
5. 山形県
6. 福島県

何か他に知りたいことがありますか？


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のお土産として人気のある品々を紹介します。以下にいくつかの代表的なお土産を挙げます。

1. **牛たん** - 宮城県名物の一つで、特に仙台で有名です。本格的な焼き牛たんを味わうことができます。

2. **ずんだ餅** - 絡めた枝豆のペースト（ずんだ）をのせた餅。甘くて食感が良いです。

3. **萩の月** - 定番の宮城土産の一つで、皮は薄く、中にはクリームがたっぷり詰まっています。

4. **お土産スイーツ** - 宮城には多くのスイーツがあり、特に「喜久福」などの和菓子が人気です。

5. **仙台味噌** - 料理のアクセントにぴったりな地元の味噌。

6. **せんべい** - 特に「サンリクせんべい」や「仙台薄皮饅頭」といった商品が人気です。

7. **宮城の日本酒** - 美味しい日本酒もお土産として喜ばれます。

これらのお土産は、宮城県を訪れる際にはぜひチェックしてみてください。また、情報源や詳細を知りたい方は、以下のリンクを参考にしてください。

- [宮城のお土産一覧](https://tsplus.asahi.co.jp/articles/gift/75865/)
- [仙台の名物お土産ガイド](https://travel.rakuten.co.jp/mytrip/howto/sendai-souvenir)

宮城県の魅力が詰まったお土産、ぜひ楽しんでください！


'質問:質問したのは何県だったっけ？'

質問したのは宮城県のお土産についてでした。何か他に知りたいことがあれば教えてください！

---ご利用ありがとうございました！---
